# Entregable 3

Incluye la fusión por concatenación de embeddings y transferencia de aprendizaje desde AstroM3-CLIP, con resultados comparativos frente a los modelos unimodales y la votación sobre sub50.

## 1. Preparación

In [ ]:
!git clone https://github.com/David04Bucheli/proyecto-integrador.git
%cd proyecto-integrador

In [ ]:
import os, sys, json, time, platform
from pathlib import Path

REPO_URL = "https://github.com/David04Bucheli/proyecto-integrador.git"
if not os.path.isdir("src"):
    if os.path.isdir("../src"):
        os.chdir("..")
    elif REPO_URL:
        os.system(f"git clone {REPO_URL} repo_clone")
        os.chdir("repo_clone")
    else:
        raise RuntimeError("No encuentro la carpeta src/. Ejecuta desde la raíz del repo")
print("Directorio de trabajo:", os.getcwd())

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src import config, datasets, models, engine, embeddings, runs, fusion, reporting, concat, transfer, compare, leakage

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else None
print("Dispositivo:", DEVICE, "|", GPU)
print("torch", torch.__version__, "| python", platform.python_version())

SUBSET = config.SUBSET_FORMAL
OUT = Path("outputs_e3")
for d in ("checkpoints", "predictions", "figures", "embeddings"):
    (OUT / d).mkdir(parents=True, exist_ok=True)

E2_CHECKPOINTS = Path("outputs_e2/checkpoints")
RUN_AUDIT = True
RUN_CHANNEL_ABLATION = True

In [ ]:
ds = datasets.load_subset(SUBSET)
print(ds)
label_names = ds["train"].features["label"].names
print("Clases del dataset:", label_names)
assert list(label_names) == config.CLASS_NAMES, "El orden de las clases no coincide con config.CLASS_NAMES"

### Protocolo

- **Auditoría:** huellas de metadatos, fotometría (canales 0–2) y espectro (canales 0–1); solo cuentan las que superan el control positivo. Si una huella válida encuentra objetos de validación o prueba en full/train, la transferencia no se ejecuta.
- **Unimodales:** mismos modelos y protocolo del Entregable 2 (Adam, lr 1e-3, batch 32, 20 épocas, semilla 42, modelo de la última época). Si están los checkpoints del Entregable 2 en `outputs_e2/checkpoints` se cargan; si no, se reentrenan.
- **Ablación de canales:** la misma GRU entrenada solo con los canales 0–2 de la fotometría.
- **Concatenación:** embeddings de 64 valores de cada modalidad → vector de 192 (fotometría | espectros | metadatos), estandarizado con la media y la desviación de train. Encima, un MLP. Capa oculta (0 = lineal, 128, 256), dropout (0.1, 0.3) y weight decay (0, 1e-4) se eligen con el macro-F1 de validación. Parada temprana con validación (paciencia 10, máximo 100 épocas).
- **Transferencia:** pesos de `AstroMLCore/AstroM3-CLIP` (preentrenado con la semilla 42, la misma partición) y código oficial de AstroM3 fijado al commit `7f296c2`.
    - *Congelado:* los pesos preentrenados no cambian. Se extraen sus embeddings (3 × 512) y encima se entrena la misma cabeza MLP con la misma búsqueda que la concatenación.
    - *Ajuste fino:* se actualizan todos los pesos, con fusión por promedio y una capa lineal nueva de 10 clases. Hiperparámetros publicados por los autores para sub50 con preentrenamiento (`configs/config-all-sub50-clip.yaml`). Se guarda el estado con mejor macro-F1 en validación.
    - La fotometría entra con 200 pasos: recorte aleatorio en train y central en validación y prueba, igual que el código oficial.
- **Prueba:** todas las elecciones se hacen con validación; la partición de prueba solo se muestra en la sección 7. Las diferencias entre estrategias se acompañan con un bootstrap pareado (2000 remuestreos).
- Una sola corrida (semilla 42).

## 2. Auditoría de integridad

AstroM3-CLIP se preentrenó con full/train. Antes de usarlo se busca si algún objeto de validación o prueba de sub50 está en full/train.

La huella de metadatos del Entregable 1 no sirve entre configuraciones: `preprocess.py` de AstroM3Dataset estandariza los metadatos con la media y la desviación del train de cada configuración, así que el mismo objeto tiene valores distintos en full y en sub50. En cambio, los canales 0–2 de la fotometría (tiempo, flujo y error) y los canales 0–1 del espectro (flujo y error) se normalizan por objeto y no dependen de la configuración.

Se prueban las tres huellas con varias precisiones. El control positivo decide cuál es válida: como sub50 sale de las particiones de full, una huella válida debe encontrar en full/train casi todos los objetos de sub50/train. Solo las huellas válidas cuentan para la decisión.

In [ ]:
AUDIT, VERDICT, AUDIT_OK = None, None, None
if RUN_AUDIT:
    full = datasets.load_subset(config.SUBSET_FULL)
    AUDIT = leakage.audit_table(full["train"], {"train": ds["train"], "validation": ds["validation"], "test": ds["test"]})
    del full
    display(AUDIT)
    VERDICT = leakage.decide(AUDIT)
    AUDIT_OK = VERDICT["ok"]
    print(VERDICT["message"])
    print("Huellas válidas:", VERDICT["valid"])

## 3. Modelos unimodales

In [ ]:
NAMES = ["fotometria", "espectros", "metadatos"]
FWD = {"fotometria": engine.forward_fn_photometry,
       "espectros": engine.forward_fn_spectra,
       "metadatos": engine.forward_fn_metadata}

def build(name):
    torch.manual_seed(config.SEED)
    if name == "fotometria":
        enc = models.PhotometryGRUEncoder(input_dim=len(ds["train"][0]["photometry"][0]))
    elif name == "espectros":
        enc = models.SpectraCNNEncoder(in_channels=len(ds["train"][0]["spectra"]))
    else:
        enc = models.MetadataMLPEncoder(input_dim=len(ds["train"][0]["metadata"]))
    return models.UnimodalClassifier(enc)

train_loader = datasets.make_dataloader(ds["train"], shuffle=True)
loaders = {s: datasets.make_dataloader(ds[s], shuffle=False) for s in ("train", "validation", "test")}
MODELS, COST = {}, {}
for name in NAMES:
    model = build(name)
    ckpt = E2_CHECKPOINTS / f"{name}_{SUBSET}.pt"
    if ckpt.exists():
        info = runs.load_checkpoint_into(model, ckpt, which="last", device=DEVICE)
        origin = "checkpoint E2"
    else:
        print(f"\n===== {name} (reentrenando) =====")
        fr = runs.fit_with_checkpoints(model, train_loader, loaders["validation"], FWD[name],
                                       device=DEVICE, class_names=config.CLASS_NAMES)
        ckpt = OUT / "checkpoints" / f"{name}_{SUBSET}.pt"
        runs.save_checkpoint(ckpt, fr, extra={"seed": config.SEED, "subset": SUBSET})
        info = runs.load_checkpoint_into(model, ckpt, which="last", device=DEVICE)
        origin = "reentrenado E3"
    MODELS[name] = model
    COST[name] = {"origen": origin, "parametros_entrenables": info["n_parameters"],
                  "segundos_entrenamiento": round(info["train_seconds"], 1)}
display(pd.DataFrame(COST).T)

In [ ]:
PRED = {"validation": {}, "test": {}}
for split in PRED:
    for n in NAMES:
        PRED[split][n] = runs.predict_proba(MODELS[n], loaders[split], DEVICE, FWD[n])
        runs.save_predictions(OUT / "predictions" / f"{n}_{SUBSET}_{split}.npz", PRED[split][n])
    fusion.check_alignment([PRED[split][n]["y_true"] for n in NAMES])
Y = {s: PRED[s]["fotometria"]["y_true"] for s in PRED}
display(pd.DataFrame({n: {"exactitud (validation)": float((PRED["validation"][n]["y_pred"] == Y["validation"]).mean()),
                          "macro-F1 (validation)": compare.macro_f1(Y["validation"], PRED["validation"][n]["y_pred"])}
                      for n in NAMES}).T.round(4))

### 3b. Ablación de canales de la fotometría

Los canales 3–8 de la fotometría son constantes dentro de cada curva: son descriptores del objeto que AstroM3 movió de los metadatos a la fotometría (amplitud, periodo, estadístico de Lafler-Kinman, variabilidad pico a pico, log MAD y extensión temporal). Para separar lo que aporta la curva de lo que aportan esos descriptores, se entrena la misma GRU, con el mismo protocolo, solo con los canales 0–2.

In [ ]:
CORE_CHANNELS = 3

def forward_core(model, batch):
    logits, _ = model(batch["photometry"][..., :CORE_CHANNELS], batch["photometry_lengths"])
    return logits, batch["label"]

PRED_CORE = None
if RUN_CHANNEL_ABLATION:
    torch.manual_seed(config.SEED)
    gru_core = models.UnimodalClassifier(models.PhotometryGRUEncoder(input_dim=CORE_CHANNELS))
    FR_CORE = runs.fit_with_checkpoints(gru_core, train_loader, loaders["validation"], forward_core,
                                        device=DEVICE, class_names=config.CLASS_NAMES)
    runs.save_checkpoint(OUT / "checkpoints" / f"fotometria_canales012_{SUBSET}.pt", FR_CORE,
                         extra={"seed": config.SEED, "subset": SUBSET, "channels": [0, 1, 2]})
    PRED_CORE = {s: runs.predict_proba(gru_core, loaders[s], DEVICE, forward_core) for s in ("validation", "test")}
    print("macro-F1 de validación | 9 canales:", round(compare.macro_f1(Y["validation"], PRED["validation"]["fotometria"]["y_pred"]), 4),
          "| canales 0-2:", round(compare.macro_f1(Y["validation"], PRED_CORE["validation"]["y_pred"]), 4))

## 4. Votación (referencia del Entregable 2)

In [ ]:
PRIORITY = fusion.rank_by_validation({n: compare.macro_f1(Y["validation"], PRED["validation"][n]["y_pred"]) for n in NAMES})

def vote(split, rule):
    return fusion.hard_vote({n: PRED[split][n]["y_pred"] for n in NAMES}, tie_break=rule, priority=PRIORITY,
                            probs={n: PRED[split][n]["probs"] for n in NAMES})["y_pred"]

VAL_RULE = {r: compare.macro_f1(Y["validation"], vote("validation", r)) for r in fusion.TIE_BREAKS}
CHOSEN_RULE = max(VAL_RULE, key=VAL_RULE.get)
VOTE = {s: vote(s, CHOSEN_RULE) for s in Y}
SOFT = {s: fusion.soft_vote({n: PRED[s][n]["probs"] for n in NAMES}) for s in Y}
print("Macro-F1 de validación por regla:", {k: round(v, 4) for k, v in VAL_RULE.items()}, "| elegida:", CHOSEN_RULE)

## 5. Fusión por concatenación de embeddings

Los embeddings de entrenamiento salen de codificadores entrenados con esos mismos objetos, así que son más limpios que los de validación y prueba. Por eso la cabeza se elige y se detiene con validación.

In [ ]:
EMB = {}
for s in ("train", "validation", "test"):
    parts = {n: embeddings.extract_embeddings(MODELS[n], loaders[s], DEVICE, FWD[n]) for n in NAMES}
    fusion.check_alignment([parts[n]["labels"] for n in NAMES])
    EMB[s] = embeddings.concatenate_modalities(parts["fotometria"]["embeddings"], parts["espectros"]["embeddings"],
                                               parts["metadatos"]["embeddings"], parts["fotometria"]["labels"])
    embeddings.save_embeddings(str(OUT / "embeddings" / f"fused_{SUBSET}_{s}.npz"), EMB[s]["embeddings"], EMB[s]["labels"])
for s in ("validation", "test"):
    assert np.array_equal(EMB[s]["labels"], Y[s])

In [ ]:
GS_CONCAT = concat.grid_search(EMB["train"]["embeddings"], EMB["train"]["labels"],
                               EMB["validation"]["embeddings"], EMB["validation"]["labels"], device=DEVICE)
CONCAT = GS_CONCAT["best"]
display(pd.DataFrame(GS_CONCAT["table"]).sort_values("val_macro_f1", ascending=False).round(4))
print("Configuración elegida:", CONCAT["params"], "| mejor época:", CONCAT["best_epoch"],
      "| parámetros:", CONCAT["n_parameters"])
concat.save_head(OUT / "checkpoints" / f"concat_{SUBSET}.pt", CONCAT, extra={"seed": config.SEED, "subset": SUBSET})
PRED_CONCAT = {s: concat.predict_proba_head(CONCAT, EMB[s]["embeddings"], EMB[s]["labels"], DEVICE) for s in Y}
reporting.plot_history({"concatenación": CONCAT["history"]}, path=OUT / "figures" / "curvas_concatenacion.png")
plt.show()

In [ ]:
COMBOS = [("fotometria",), ("fotometria", "metadatos"), ("fotometria", "espectros"),
          ("espectros", "metadatos"), tuple(NAMES)]
best_hp = {k: CONCAT["params"][k] for k in ("hidden_dim", "dropout", "weight_decay")}
ABLATION = {}
for combo in COMBOS:
    X = {s: concat.take_modalities(EMB[s]["embeddings"], combo, config.EMBEDDING_DIM) for s in EMB}
    fit = concat.train_head(X["train"], EMB["train"]["labels"], X["validation"], EMB["validation"]["labels"],
                            device=DEVICE, **best_hp)
    ABLATION[" + ".join(combo)] = {"macro-F1 (validation)": compare.macro_f1(
        Y["validation"], concat.predict_proba_head(fit, X["validation"], device=DEVICE)["y_pred"]),
        "mejor época": fit["best_epoch"]}
ABLATION = pd.DataFrame(ABLATION).T.round(4)
display(ABLATION)

## 6. Transferencia desde AstroM3-CLIP

Se descarga el código oficial de AstroM3 (fijado a un commit) y los pesos desde Hugging Face. La celda se detiene si la auditoría encontró coincidencias.

In [ ]:
assert AUDIT_OK is not False, "La auditoría encontró coincidencias: no se usa AstroM3-CLIP"
if AUDIT_OK is None:
    print("Aviso: la auditoría no se ejecutó en esta sesión")
ROOT = transfer.fetch_astrom3_code()
CACHE = {s: transfer.CachedSplit(ds[s]) for s in ("train", "validation", "test")}
print({s: len(v) for s, v in CACHE.items()})

### 6a. Modelo preentrenado congelado

In [ ]:
clip_enc, clip_info = transfer.load_clip_encoder(config.CLIP_MODEL_NAME, ROOT)
print("Tensores cargados:", clip_info["n_tensors"], "| faltantes:", clip_info["missing"], "| sobrantes:", clip_info["unexpected"])
print("Parámetros del modelo preentrenado:", f"{sum(p.numel() for p in clip_enc.parameters()):,}")

t0 = time.time()
CLIP_EMB = {}
for s in CACHE:
    e = transfer.extract_clip_embeddings(clip_enc, transfer.make_clip_loader(CACHE[s], train=False, batch_size=256), DEVICE)
    assert np.array_equal(e["labels"], EMB[s]["labels"])
    CLIP_EMB[s] = {"embeddings": np.concatenate([e[n] for n in NAMES], axis=1), "labels": e["labels"]}
    np.savez(OUT / "embeddings" / f"clip_{SUBSET}_{s}.npz", **CLIP_EMB[s])
CLIP_EXTRACT_SECONDS = time.time() - t0
print("Embeddings AstroM3-CLIP:", CLIP_EMB["train"]["embeddings"].shape, f"| extracción: {CLIP_EXTRACT_SECONDS:.0f} s")
del clip_enc
if DEVICE == "cuda":
    torch.cuda.empty_cache()

In [ ]:
GS_FROZEN = concat.grid_search(CLIP_EMB["train"]["embeddings"], CLIP_EMB["train"]["labels"],
                               CLIP_EMB["validation"]["embeddings"], CLIP_EMB["validation"]["labels"], device=DEVICE)
FROZEN = GS_FROZEN["best"]
display(pd.DataFrame(GS_FROZEN["table"]).sort_values("val_macro_f1", ascending=False).round(4))
print("Configuración elegida:", FROZEN["params"], "| mejor época:", FROZEN["best_epoch"],
      "| parámetros:", FROZEN["n_parameters"])
concat.save_head(OUT / "checkpoints" / f"astrom3_congelado_{SUBSET}.pt", FROZEN, extra={"seed": config.SEED, "subset": SUBSET})
PRED_FROZEN = {s: concat.predict_proba_head(FROZEN, CLIP_EMB[s]["embeddings"], CLIP_EMB[s]["labels"], DEVICE) for s in Y}

### 6b. Ajuste fino de todo el modelo

In [ ]:
torch.manual_seed(config.SEED)
HP = dict(transfer.FINETUNE_HP)
ft_model, ft_info = transfer.build_finetune_model(config.CLIP_MODEL_NAME, ROOT, hp=HP)
print("Capas nuevas:", ft_info["new_layers"])
print("Hiperparámetros:", HP)
ft_train_loader = transfer.make_clip_loader(CACHE["train"], train=True, batch_size=HP["batch_size"])
CLIP_LOADERS = {s: transfer.make_clip_loader(CACHE[s], train=False, batch_size=256) for s in ("validation", "test")}
FT = transfer.finetune(ft_model, ft_train_loader, CLIP_LOADERS["validation"], DEVICE, hp=HP, class_names=config.CLASS_NAMES)
runs.save_checkpoint(OUT / "checkpoints" / f"astrom3_ajuste_fino_{SUBSET}.pt", FT,
                     extra={"seed": config.SEED, "subset": SUBSET, "hp": HP, "astrom3_commit": transfer.ASTROM3_COMMIT})
print(f"Mejor época (macro-F1 de validación): {FT['best_epoch']} de {FT['stopped_epoch']} | {FT['train_seconds']:.0f} s | {FT['n_parameters']:,} parámetros entrenables")
reporting.plot_history({"AstroM3 ajuste fino": FT["history"]}, path=OUT / "figures" / "curvas_astrom3_ajuste_fino.png")
plt.show()

In [ ]:
PRED_FT = {s: runs.predict_proba(ft_model, CLIP_LOADERS[s], DEVICE, transfer.forward_fn_clip) for s in Y}
for s in Y:
    assert np.array_equal(PRED_FT[s]["y_true"], Y[s])
    runs.save_predictions(OUT / "predictions" / f"astrom3_ajuste_fino_{SUBSET}_{s}.npz", PRED_FT[s])

## 7. Comparación de estrategias

In [ ]:
def entry(preds, params, seconds):
    return {**{s: {"y_true": Y[s], "y_pred": preds[s]} for s in Y},
            "parametros_entrenables": int(params), "segundos_entrenamiento": round(float(seconds), 1)}

LABELS = {"fotometria": "GRU (fotometría)", "espectros": "CNN 1D (espectros)", "metadatos": "MLP (metadatos)"}
ENTRIES = {LABELS[n]: entry({s: PRED[s][n]["y_pred"] for s in Y}, COST[n]["parametros_entrenables"],
                            COST[n]["segundos_entrenamiento"]) for n in NAMES}
ENTRIES[f"Votación 2 de 3 ({CHOSEN_RULE})"] = entry(VOTE, 0, 0)
if PRED_CORE is not None:
    ENTRIES["GRU (fotometría, canales 0-2)"] = entry({s: PRED_CORE[s]["y_pred"] for s in Y}, FR_CORE["n_parameters"],
                                                     FR_CORE["train_seconds"])
ENTRIES["Voto suave (referencia)"] = entry(SOFT, 0, 0)
ENTRIES["Concatenación + MLP"] = entry({s: PRED_CONCAT[s]["y_pred"] for s in Y}, CONCAT["n_parameters"], CONCAT["train_seconds"])
ENTRIES["AstroM3-CLIP congelado + MLP"] = entry({s: PRED_FROZEN[s]["y_pred"] for s in Y}, FROZEN["n_parameters"],
                                                FROZEN["train_seconds"] + CLIP_EXTRACT_SECONDS)
ENTRIES["AstroM3-CLIP ajuste fino"] = entry({s: PRED_FT[s]["y_pred"] for s in Y}, FT["n_parameters"], FT["train_seconds"])

TABLE = compare.results_table(ENTRIES).round(4)
display(TABLE)
TABLE.to_csv(OUT / "comparacion_estrategias.csv")
compare.plot_strategy_bars(TABLE, path=OUT / "figures" / "comparacion_macro_f1_test.png",
                           title=f"Macro-F1 en prueba ({SUBSET}, semilla {config.SEED})")
plt.show()

Para las fusiones construidas sobre los modelos unimodales (votación, voto suave y concatenación), los parámetros y segundos son los adicionales a los de los tres codificadores.

Diferencias en prueba con bootstrap pareado: Δ es la estrategia de la fila menos la referencia; si el intervalo del 95 % incluye el 0, la diferencia no es concluyente con esta partición.

In [ ]:
TEST_PREDS = {name: e["test"]["y_pred"] for name, e in ENTRIES.items()}
BOOT_PHOT = compare.bootstrap_table(Y["test"], TEST_PREDS, reference="GRU (fotometría)")
BOOT_CONCAT = compare.bootstrap_table(Y["test"], TEST_PREDS, reference="Concatenación + MLP")
print("Referencia: GRU (fotometría)")
display(BOOT_PHOT)
print("Referencia: Concatenación + MLP")
display(BOOT_CONCAT)

In [ ]:
SUMMARY = {split: {name: fusion.summarize_predictions(Y[split], e[split]["y_pred"], config.CLASS_NAMES)
                   for name, e in ENTRIES.items()} for split in Y}
F1_CLASS = {split: reporting.per_class_table(SUMMARY[split], "f1").round(3) for split in Y}
for split in Y:
    print(f"F1 por clase ({split})")
    display(F1_CLASS[split])
with open(OUT / "reportes_por_clase.txt", "w", encoding="utf-8") as f:
    for split in Y:
        for name, m in SUMMARY[split].items():
            f.write(f"===== {name} | {split} =====\n{m['report_text']}\n")
            reporting.plot_confusion_matrix(m["confusion_matrix"], title=f"{name} — {split} {SUBSET} (recall por fila)",
                                            path=OUT / "figures" / f"cm_{compare.slug(name)}_{SUBSET}_{split}.png")
            plt.close("all")
print("Reportes por clase y matrices de validación y prueba guardados en", OUT)

In [ ]:
for name in ("Concatenación + MLP", "AstroM3-CLIP congelado + MLP", "AstroM3-CLIP ajuste fino"):
    for split in Y:
        print(f"\n===== {name} | {split} =====")
        print(SUMMARY[split][name]["report_text"])
        reporting.plot_confusion_matrix(SUMMARY[split][name]["confusion_matrix"], title=f"{name} — {split} {SUBSET} (recall por fila)")
        plt.show()

## 8. Resumen guardado

In [ ]:
def clean(m):
    return {"accuracy": float(m["accuracy"]), "macro_f1": float(m["macro_f1"]),
            "per_class_f1": {c: float(v["f1"]) for c, v in m["per_class"].items()}}

results = {
    "subset": SUBSET, "seed": config.SEED, "device": DEVICE, "gpu": GPU, "torch": torch.__version__,
    "astrom3_commit": transfer.ASTROM3_COMMIT, "clip_model": config.CLIP_MODEL_NAME,
    "audit": AUDIT.reset_index().to_dict(orient="records") if AUDIT is not None else None,
    "audit_verdict": VERDICT,
    "unimodal_cost": COST,
    "vote": {"chosen_tie_break": CHOSEN_RULE, "val_macro_f1_by_rule": VAL_RULE},
    "concat": {"grid": GS_CONCAT["table"], "chosen": CONCAT["params"], "best_epoch": CONCAT["best_epoch"],
               "ablation": ABLATION.to_dict(orient="index")},
    "astrom3_frozen": {"grid": GS_FROZEN["table"], "chosen": FROZEN["params"], "best_epoch": FROZEN["best_epoch"],
                       "extract_seconds": CLIP_EXTRACT_SECONDS},
    "astrom3_finetune": {"hp": HP, "best_epoch": FT["best_epoch"], "stopped_epoch": FT["stopped_epoch"],
                         "history": FT["history"]},
    "comparison": TABLE.to_dict(orient="index"),
    "bootstrap_vs_photometry": BOOT_PHOT.to_dict(orient="index"),
    "bootstrap_vs_concat": BOOT_CONCAT.to_dict(orient="index"),
    "per_split": {split: {name: clean(m) for name, m in SUMMARY[split].items()} for split in Y},
}
with open(OUT / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("Guardado:", OUT / "results.json")